In [1]:
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Dense, Dropout, LSTM, Conv1D, MaxPooling1D, 
                                     Flatten, Input, MultiHeadAttention,
                                     LayerNormalization, GlobalAveragePooling1D)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K
import joblib
from matplotlib import pyplot as plt


In [2]:
# ================================================================
# 1️⃣ Paths
# ================================================================
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "..", "data", "processed_data")
MODEL_DIR = os.path.join(BASE_DIR, "..", "models")
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"📁 Loading processed data from: {DATA_DIR}")
print(f"💾 Models will be saved to: {MODEL_DIR}")

📁 Loading processed data from: c:\Users\DELL\Documents\PRAXIS\Projects\Capstone_Project_final\backend\utils\..\data\processed_data
💾 Models will be saved to: c:\Users\DELL\Documents\PRAXIS\Projects\Capstone_Project_final\backend\utils\..\models


In [3]:
# ================================================================
# 2️⃣ Utility: Build Models
# ================================================================
output_window = 5

def build_lstm(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        LSTM(64, return_sequences=True),
        Dropout(0.15),
        LSTM(64),
        Dropout(0.15),
        Dense(32, activation='relu'),
        Dense(output_window)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    return model

def build_cnn(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(64, 3, activation='relu'),
        MaxPooling1D(2),
        Dropout(0.15),
        Conv1D(32, 2, activation='relu'),
        Flatten(),
        Dense(32, activation='relu'),
        Dense(output_window)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    return model

def build_hybrid(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(64, 3, activation='relu'),
        MaxPooling1D(2),
        Dropout(0.15),
        LSTM(64),
        Dropout(0.15),
        Dense(32, activation='relu'),
        Dense(output_window)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    return model

# --- Transformer ---
def transformer_block(x, head_size, num_heads, ff_dim, dropout=0.1):
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=head_size)(x, x)
    attn_output = Dropout(dropout)(attn_output)
    out1 = LayerNormalization(epsilon=1e-6)(x + attn_output)

    ffn = Dense(ff_dim, activation="relu")(out1)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout)(ffn)

    return LayerNormalization(epsilon=1e-6)(out1 + ffn)

def build_transformer_model(input_shape, head_size=32, num_heads=2, ff_dim=32, dropout=0.1):
    inputs = Input(shape=input_shape)
    x = transformer_block(inputs, head_size, num_heads, ff_dim, dropout)
    x = GlobalAveragePooling1D()(x)
    x = Dense(32, activation='relu')(x)
    outputs = Dense(output_window)(x)
    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model

In [4]:
models = {
    "CNN": build_cnn,
    "LSTM": build_lstm,
    "Hybrid": build_hybrid,
    "Transformer": build_transformer_model
}

In [5]:
# ================================================================
# 3️⃣ Function: Train & Evaluate
# ================================================================
def train_and_evaluate(X_train, y_train, X_test, y_test, scaler, sector, stock):
    input_shape = (X_train.shape[1], X_train.shape[2])
    sector_dir = os.path.join(MODEL_DIR, sector)
    os.makedirs(sector_dir, exist_ok=True)
    stock_dir = os.path.join(sector_dir, stock)
    os.makedirs(stock_dir, exist_ok=True)

    metrics = {}

    for name, build_fn in models.items():
        print(f"\n🚀 Training {name} model for {stock} ({sector})")
        K.clear_session()

        # --- Early stopping ---
        es = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

        # --- Build model ---
        model = build_fn(input_shape)

        # --- Train ---
        history = model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=20,
            batch_size=16,
            verbose=0,
            callbacks=[es]
        )

        # --- Predict ---
        y_pred = model.predict(X_test, verbose=0)

        # --- Reshape to 2D for inverse scaling ---
        y_pred_2d = y_pred.reshape(-1, 1)
        y_test_2d = y_test.reshape(-1, 1)

        # --- Inverse transform ---
        y_pred_inv = scaler.inverse_transform(y_pred_2d)
        y_true_inv = scaler.inverse_transform(y_test_2d)

        # --- Reshape back ---
        y_pred_inv = y_pred_inv.reshape(y_pred.shape)
        y_true_inv = y_true_inv.reshape(y_test.shape)

        # --- Metrics ---
        rmse = np.sqrt(mean_squared_error(y_true_inv.flatten(), y_pred_inv.flatten()))
        mae = mean_absolute_error(y_true_inv.flatten(), y_pred_inv.flatten())
        mean_val = np.mean(y_true_inv.flatten())
        rmse_mean_ratio = rmse / mean_val if mean_val != 0 else np.nan

        print(f"   ✅ {name} RMSE: {rmse:.3f}, MAE: {mae:.3f}, RMSE/Mean: {rmse_mean_ratio:.4f}")

        metrics[name] = {
            "RMSE": round(rmse, 4),
            "MAE": round(mae, 4),
            "RMSE/Mean": round(rmse_mean_ratio, 4)
        }

        # --- Save model ---
        model_path = os.path.join(stock_dir, f"{name}.keras")
        model.save(model_path)

        print(f"   💾 Saved model -> {model_path}")

    # --- Save metrics JSON ---
    metrics_path = os.path.join(stock_dir, "metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=4)
    print(f"📊 Metrics saved -> {metrics_path}")


In [6]:
# ================================================================
# 4️⃣ Loop through sectors and stocks (robust scaler + model saving)
# ================================================================
import os
import numpy as np
import joblib

# Ensure model directory exists
os.makedirs(MODEL_DIR, exist_ok=True)

for sector in os.listdir(DATA_DIR):
    sector_path = os.path.join(DATA_DIR, sector)
    if not os.path.isdir(sector_path):
        continue

    print(f"\n📂 Processing sector: {sector}")

    for file in os.listdir(sector_path):
        if file.endswith(".npz"):
            stock_name = file.replace(".npz", "")
            npz_path = os.path.join(sector_path, file)

            # --- Load preprocessed data ---
            try:
                data = np.load(npz_path)
                X_train, y_train = data["X_train"], data["y_train"]
                X_test, y_test = data["X_test"], data["y_test"]
            except Exception as e:
                print(f"❌ Error loading data for {stock_name}: {e}")
                continue

            # --- Locate scaler file ---
            possible_names = [
                f"{stock_name}-scaler.pkl",
                f"{stock_name}.scaler.pkl",
                f"{stock_name}_scaler.pkl",
                f"{stock_name.replace('-', '_')}-scaler.pkl",
            ]

            scaler_path = None
            for name in possible_names:
                path = os.path.join(sector_path, name)
                if os.path.exists(path):
                    scaler_path = path
                    break

            if not scaler_path:
                print(f"⚠️ Scaler file not found for {stock_name} in {sector_path}")
                continue

            # --- Load scaler ---
            try:
                scaler = joblib.load(scaler_path)
            except Exception as e:
                print(f"❌ Error loading scaler for {stock_name}: {e}")
                continue

            # --- Train and evaluate all 4 models ---
            print(f"🚀 Starting training for {stock_name} ({sector})")
            train_and_evaluate(X_train, y_train, X_test, y_test, scaler, sector, stock_name)

print("\n🎯 Training complete. All models saved successfully.")


📂 Processing sector: Auto
🚀 Starting training for BAJAJ-AUTO (Auto)

🚀 Training CNN model for BAJAJ-AUTO (Auto)

   ✅ CNN RMSE: 271.676, MAE: 211.813, RMSE/Mean: 0.0315
   💾 Saved model -> c:\Users\DELL\Documents\PRAXIS\Projects\Capstone_Project_final\backend\utils\..\models\Auto\BAJAJ-AUTO\CNN.keras

🚀 Training LSTM model for BAJAJ-AUTO (Auto)
   ✅ LSTM RMSE: 321.753, MAE: 249.624, RMSE/Mean: 0.0373
   💾 Saved model -> c:\Users\DELL\Documents\PRAXIS\Projects\Capstone_Project_final\backend\utils\..\models\Auto\BAJAJ-AUTO\LSTM.keras

🚀 Training Hybrid model for BAJAJ-AUTO (Auto)
   ✅ Hybrid RMSE: 250.065, MAE: 189.762, RMSE/Mean: 0.0290
   💾 Saved model -> c:\Users\DELL\Documents\PRAXIS\Projects\Capstone_Project_final\backend\utils\..\models\Auto\BAJAJ-AUTO\Hybrid.keras

🚀 Training Transformer model for BAJAJ-AUTO (Auto)
   ✅ Transformer RMSE: 2257.114, MAE: 2228.547, RMSE/Mean: 0.2620
   💾 Saved model -> c:\Users\DELL\Documents\PRAXIS\Projects\Capstone_Project_final\backend\utils\..\m

In [7]:
# ================================================================
# 1️⃣ Evaluation & Metrics Collection (Enhanced)
# ================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from keras.models import load_model
from sklearn.metrics import mean_squared_error, mean_absolute_error

DATA_DIR = "../data/processed_data"
MODEL_DIR = "../models"
OUTPUT_DIR = "../output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [8]:
metrics_records = []

# --- Loop through all sectors and stocks ---
for sector in os.listdir(MODEL_DIR):
    sector_path = os.path.join(MODEL_DIR, sector)
    if not os.path.isdir(sector_path):
        continue

    print(f"\n📊 Evaluating sector: {sector}")

    for stock in os.listdir(sector_path):
        stock_path = os.path.join(sector_path, stock)
        if not os.path.isdir(stock_path):
            continue

        npz_path = os.path.join(DATA_DIR, sector, f"{stock}.npz")
        scaler_path = os.path.join(DATA_DIR, sector, f"{stock}_scaler.pkl")
        if not os.path.exists(npz_path) or not os.path.exists(scaler_path):
            print(f"⚠️ Missing data/scaler for {stock} in {sector}")
            continue

        # --- Load data & scaler ---
        data = np.load(npz_path)
        X_test, y_test, X_forecast = data["X_test"], data["y_test"], data["X_forecast"]
        scaler = joblib.load(scaler_path)

        # --- Inverse transform true values ---
        y_test_2d = y_test.reshape(-1, 1)
        y_true_inv = scaler.inverse_transform(y_test_2d).reshape(y_test.shape)
        mean_true = np.mean(y_true_inv)

        # --- Prepare plot (only last 100 points) ---
        N = min(100, len(y_true_inv.flatten()))
        plt.figure(figsize=(10, 5))
        plt.plot(y_true_inv.flatten()[-N:], label="Actual", color="black", linewidth=2)

        # --- Temporary store metrics per model ---
        model_metrics = {}

        for model_file in os.listdir(stock_path):
            if model_file.endswith(".keras"):
                model_name = model_file.replace(".keras", "")
                model_path = os.path.join(stock_path, model_file)

                try:
                    model = load_model(model_path, compile=False)
                    y_pred = model.predict(X_test, verbose=0)

                    # Inverse transform predictions
                    y_pred_2d = y_pred.reshape(-1, 1)
                    y_pred_inv = scaler.inverse_transform(y_pred_2d).reshape(y_pred.shape)

                    rmse = np.sqrt(mean_squared_error(y_true_inv.flatten(), y_pred_inv.flatten()))
                    mae = mean_absolute_error(y_true_inv.flatten(), y_pred_inv.flatten())
                    rmse_mean = rmse / mean_true

                    model_metrics[model_name] = {
                        "RMSE": round(rmse, 3),
                        "MAE": round(mae, 3),
                        "RMSE/Mean": round(rmse_mean, 4)
                    }

                    # Plot last N points only
                    plt.plot(y_pred_inv.flatten()[-N:], label=f"{model_name}", alpha=0.8)

                except Exception as e:
                    print(f"❌ Error evaluating {model_name} for {stock}: {e}")

        # --- Find best models ---
        if model_metrics:
            best_rmse_model = min(model_metrics, key=lambda m: model_metrics[m]["RMSE"])
            best_rmse_mean_model = min(model_metrics, key=lambda m: model_metrics[m]["RMSE/Mean"])

            # --- Combine all results into one row ---
            row = {
                "Sector": sector,
                "Stock": stock,
                "Best_by_RMSE": best_rmse_model,
                "Best_by_RMSE/Mean": best_rmse_mean_model
            }

            for model_name, vals in model_metrics.items():
                row[f"{model_name}_RMSE"] = vals["RMSE"]
                row[f"{model_name}_MAE"] = vals["MAE"]
                row[f"{model_name}_RMSE/Mean"] = vals["RMSE/Mean"]

            metrics_records.append(row)

        # --- Save evaluation plot ---
        plt.title(f"{sector} - {stock}: Last {N} Test Points")
        plt.xlabel("Time Steps")
        plt.ylabel("Stock Price")
        plt.legend()
        sector_out = os.path.join(OUTPUT_DIR, sector)
        os.makedirs(sector_out, exist_ok=True)
        plt.savefig(os.path.join(sector_out, f"{stock}_evaluation.png"))
        plt.close()

# --- Convert to DataFrame and save ---
metrics_df = pd.DataFrame(metrics_records)
metrics_csv_path = os.path.join(OUTPUT_DIR, "model_metrics_summary.csv")
metrics_df.to_csv(metrics_csv_path, index=False)

print(f"\n✅ Evaluation complete! Metrics saved -> {metrics_csv_path}")
display(metrics_df.head(10))



📊 Evaluating sector: Auto

📊 Evaluating sector: Chemicals

📊 Evaluating sector: FMCG

📊 Evaluating sector: Healthcare

📊 Evaluating sector: IT

📊 Evaluating sector: Media

📊 Evaluating sector: Metal

📊 Evaluating sector: OilGas

📊 Evaluating sector: Pharma

📊 Evaluating sector: PrivateBank

📊 Evaluating sector: PSUBank

✅ Evaluation complete! Metrics saved -> ../output\model_metrics_summary.csv


,Sector,Stock,Best_by_RMSE,Best_by_RMSE/Mean,CNN_RMSE,CNN_MAE,CNN_RMSE/Mean,Hybrid_RMSE,Hybrid_MAE,Hybrid_RMSE/Mean,LSTM_RMSE,LSTM_MAE,LSTM_RMSE/Mean,Transformer_RMSE,Transformer_MAE,Transformer_RMSE/Mean
0,Auto,BAJAJ-AUTO,Hybrid,Hybrid,271.676,211.813,0.0315,250.065,189.762,0.0290,321.753,249.624,0.0373,2257.114,2228.547,0.2620
1,Auto,EICHERMOT,Hybrid,Hybrid,249.843,190.604,0.0411,220.928,171.101,0.0364,349.388,282.456,0.0575,2426.058,2327.550,0.3992
2,Auto,M&M,CNN,CNN,112.442,84.348,0.0340,173.305,141.193,0.0523,240.950,202.663,0.0728,1522.234,1508.144,0.4597
3,Auto,MARUTI,CNN,CNN,509.082,360.415,0.0365,963.865,744.587,0.0691,917.296,702.606,0.0657,4244.184,3900.930,0.3041
4,Auto,TATAMOTORS,Hybrid,Hybrid,76.804,37.008,0.1182,71.578,35.387,0.1101,73.149,38.572,0.1126,103.206,52.147,0.1588
5,Chemicals,PIDILITIND,Hybrid,Hybrid,34.157,27.258,0.0227,32.673,26.315,0.0217,38.258,30.856,0.0254,190.792,187.955,0.1265
6,Chemicals,PIIND,CNN,CNN,124.180,92.999,0.0322,141.677,112.046,0.0367,141.477,110.421,0.0367,381.404,308.495,0.0989
7,Chemicals,SOLARINDS,CNN,CNN,731.273,563.845,0.0486,754.638,588.752,0.0501,1454.838,1190.911,0.0966,8522.427,8437.773,0.5661
8,Chemicals,SRF,CNN,CNN,142.760,112.205,0.0473,155.931,128.508,0.0516,167.256,138.144,0.0554,679.651,668.299,0.2250
9,Chemicals,UPL,LSTM,LSTM,23.131,17.280,0.0340,29.616,23.044,0.0436,23.065,18.025,0.0339,119.728,115.498,0.1761


In [9]:
# ================================================================
# ✅ Final Stable Version — Forecast Next 5 Trading Days
# Handles weekends, holidays, and unequal prediction lengths
# ================================================================
from datetime import datetime, timedelta
import yfinance as yf
import pandas as pd
import numpy as np
import os
import joblib
from tensorflow.keras.models import load_model

forecast_records = []

for sector in os.listdir(MODEL_DIR):
    sector_path = os.path.join(MODEL_DIR, sector)
    if not os.path.isdir(sector_path):
        continue

    for stock in os.listdir(sector_path):
        stock_path = os.path.join(sector_path, stock)
        if not os.path.isdir(stock_path):
            continue

        npz_path = os.path.join(DATA_DIR, sector, f"{stock}.npz")
        scaler_path = os.path.join(DATA_DIR, sector, f"{stock}_scaler.pkl")
        if not os.path.exists(npz_path) or not os.path.exists(scaler_path):
            continue

        # --- Load data and scaler ---
        data = np.load(npz_path)
        X_forecast = data["X_forecast"]
        scaler = joblib.load(scaler_path)
        forecast_dict = {"Sector": sector, "Stock": stock}

        # --- Get last available trading date ---
        stock_data_path = os.path.join("../data/stock_data", sector, f"{stock}.csv")
        if os.path.exists(stock_data_path):
            try:
                df_hist = pd.read_csv(stock_data_path)
                df_hist.columns = [c.lower() for c in df_hist.columns]
                if "date" in df_hist.columns:
                    df_hist["date"] = pd.to_datetime(df_hist["date"])
                    last_date = df_hist["date"].max()
                else:
                    last_date = pd.Timestamp.today()
            except Exception:
                last_date = pd.Timestamp.today()
        else:
            last_date = pd.Timestamp.today()

        # --- Fetch next valid 5 trading days ---
        # --- Generate next 5 valid trading days (skip weekends) ---
        forecast_dates = []
        current_date = last_date + timedelta(days=1)
        while len(forecast_dates) < 5:
            # Monday–Friday = valid market day
            if current_date.weekday() < 5:
                forecast_dates.append(current_date.strftime("%d-%m-%Y"))
            current_date += timedelta(days=1)

        # ✅ Guarantee 5 dates by padding if fewer found
        if len(forecast_dates) < 5:
            last_used_date = pd.to_datetime(forecast_dates[-1]) if forecast_dates else pd.Timestamp.today()
            extra_dates = pd.bdate_range(start=last_used_date + timedelta(days=1), periods=5 - len(forecast_dates))
            forecast_dates.extend(extra_dates.strftime("%d-%m-%Y").tolist())
        forecast_dates = forecast_dates[:5]

        # --- Model predictions ---
        for model_file in os.listdir(stock_path):
            if model_file.endswith(".keras"):
                model_name = model_file.replace(".keras", "")
                model_path = os.path.join(stock_path, model_file)
                try:
                    model = load_model(model_path, compile=False)
                    y_forecast = model.predict(X_forecast, verbose=0)
                    y_forecast_inv = scaler.inverse_transform(
                        y_forecast.reshape(-1, 1)
                    ).flatten()

                    # ✅ Ensure exactly 5 values
                    if len(y_forecast_inv) > 5:
                        y_forecast_inv = y_forecast_inv[:5]
                    elif len(y_forecast_inv) < 5:
                        y_forecast_inv = np.pad(
                            y_forecast_inv, (0, 5 - len(y_forecast_inv)), mode="edge"
                        )

                    forecast_dict[model_name] = list(np.round(y_forecast_inv, 2))

                except Exception as e:
                    print(f"❌ Forecast error for {stock} ({model_name}): {e}")

        # --- Build final 5-day forecast safely ---
        try:
            df_forecast = pd.DataFrame({
                "Date": forecast_dates[:5],
                "LSTM": forecast_dict.get("LSTM", [None]*5)[:5],
                "CNN": forecast_dict.get("CNN", [None]*5)[:5],
                "Hybrid": forecast_dict.get("Hybrid", [None]*5)[:5],
                "Transformer": forecast_dict.get("Transformer", [None]*5)[:5],
            })

            print(f"\n📈 Next 5-Day Forecast for {stock} ({sector}):")
            print(df_forecast)

            for i, row in df_forecast.iterrows():
                forecast_records.append({
                    "Sector": sector,
                    "Stock": stock,
                    "Date": row["Date"],
                    "CNN": row["CNN"],
                    "LSTM": row["LSTM"],
                    "Hybrid": row["Hybrid"],
                    "Transformer": row["Transformer"],
                })

        except Exception as e:
            print(f"❌ Skipped {stock} ({sector}) due to DataFrame error: {e}")

# --- Save combined forecasts ---
forecast_df = pd.DataFrame(forecast_records)
forecast_csv_path = os.path.join(OUTPUT_DIR, "forecast_5days.csv")
forecast_df.to_csv(forecast_csv_path, index=False)

print(f"\n✅ 5-Day Forecasts saved -> {forecast_csv_path}")
forecast_df.head(10)



📈 Next 5-Day Forecast for BAJAJ-AUTO (Auto):
         Date         LSTM          CNN       Hybrid  Transformer
0  07-11-2025  8852.160156  8734.639648  8726.269531  6372.859863
1  10-11-2025  8854.320312  8733.179688  8869.240234  6380.759766
2  11-11-2025  8703.190430  8856.190430  8773.049805  6387.870117
3  12-11-2025  9131.209961  8983.049805  8802.120117  6395.049805
4  13-11-2025  8729.629883  8815.919922  8807.669922  6401.379883

📈 Next 5-Day Forecast for EICHERMOT (Auto):
         Date         LSTM          CNN       Hybrid  Transformer
0  07-11-2025  6509.879883  6774.490234  6914.129883  3744.860107
1  10-11-2025  6518.919922  6870.009766  6842.359863  3746.820068
2  11-11-2025  6625.729980  6910.919922  6801.640137  3749.209961
3  12-11-2025  6515.259766  6783.379883  6773.620117  3751.870117
4  13-11-2025  6574.180176  6781.600098  6834.290039  3754.100098

📈 Next 5-Day Forecast for M&M (Auto):
         Date         LSTM          CNN       Hybrid  Transformer
0  07-11-202

,Sector,Stock,Date,CNN,LSTM,Hybrid,Transformer
0,Auto,BAJAJ-AUTO,07-11-2025,8734.639648,8852.160156,8726.269531,6372.859863
1,Auto,BAJAJ-AUTO,10-11-2025,8733.179688,8854.320312,8869.240234,6380.759766
2,Auto,BAJAJ-AUTO,11-11-2025,8856.190430,8703.190430,8773.049805,6387.870117
3,Auto,BAJAJ-AUTO,12-11-2025,8983.049805,9131.209961,8802.120117,6395.049805
4,Auto,BAJAJ-AUTO,13-11-2025,8815.919922,8729.629883,8807.669922,6401.379883
5,Auto,EICHERMOT,07-11-2025,6774.490234,6509.879883,6914.129883,3744.860107
6,Auto,EICHERMOT,10-11-2025,6870.009766,6518.919922,6842.359863,3746.820068
7,Auto,EICHERMOT,11-11-2025,6910.919922,6625.729980,6801.640137,3749.209961
8,Auto,EICHERMOT,12-11-2025,6783.379883,6515.259766,6773.620117,3751.870117
9,Auto,EICHERMOT,13-11-2025,6781.600098,6574.180176,6834.290039,3754.100098
